In [1]:
#Google Drive video link
#
# Github link
# 

# AI CA2

**Student name:** Telmuun Dunia  
**Student number:** 2023306  
**Module:** AI CA2 
**Lecturer:** David McQuaid


**What's inside**

1. The data
2. Part 1 – Solving it as a CSP


## Setup


In [2]:
import time
import constraint
from ortools.sat.python import cp_model

## 1. The data

In [3]:
BOXES = {
 "B1":(2,30),"B2":(3,50),"B3":(1,20),"B4":(4,60),"B5":(2,40),"B6":(3,25),"B7":(1,15),"B8":(2,35),
 "B9":(4,65),"B10":(3,45),"B11":(2,30),"B12":(1,25),"B13":(3,40),"B14":(2,20),"B15":(4,70),
 "B16":(2,35),"B17":(4,55),"B18":(1,15),"B19":(3,45),"B20":(2,30),"B21":(1,20),"B22":(4,70),
 "B23":(2,40),"B24":(3,50),"B25":(1,10),"B26":(2,35),"B27":(3,55),"B28":(4,60),"B29":(2,25),"B30":(1,15)}
AREAS = {"A1":(7,18,260),"A2":(6,16,250),"A3":(7,18,270),"A4":(6,15,240),"A5":(7,17,260)}

import time
from itertools import combinations
NONE = "NONE"

print(f"{'Box':<5}{'Size':>5}{'Weight':>8}")
for b, (v, w) in BOXES.items():
    print(f"{b:<5}{v:>5}{w:>8}")
tot_v = sum(v for v, w in BOXES.values()); tot_w = sum(w for v, w in BOXES.values())
print(f"\nTotal: {len(BOXES)} boxes, {tot_v} m3, {tot_w} kg")
for label, ar in [("Scenario 1", AREAS), ("Scenario 2 (no A3)", {a: c for a, c in AREAS.items() if a != "A3"})]:
    caps = [sum(c[i] for c in ar.values()) for i in range(3)]
    print(f"{label} capacity: {caps[0]} boxes, {caps[1]} m3, {caps[2]} kg")

Box   Size  Weight
B1       2      30
B2       3      50
B3       1      20
B4       4      60
B5       2      40
B6       3      25
B7       1      15
B8       2      35
B9       4      65
B10      3      45
B11      2      30
B12      1      25
B13      3      40
B14      2      20
B15      4      70
B16      2      35
B17      4      55
B18      1      15
B19      3      45
B20      2      30
B21      1      20
B22      4      70
B23      2      40
B24      3      50
B25      1      10
B26      2      35
B27      3      55
B28      4      60
B29      2      25
B30      1      15

Total: 30 boxes, 72 m3, 1130 kg
Scenario 1 capacity: 33 boxes, 84 m3, 1280 kg
Scenario 2 (no A3) capacity: 26 boxes, 66 m3, 1010 kg


## 2. Part 1 – Solving it as a CSP
I use the two constraint libraries from the lecture notes:
* **python-constraint** – the main solver.
* **OR-Tools** – a second solver, used to double-check the answers.

### How the problem is set up
A CSP needs three things:

* **Variables:** one per box (B1 to B30). Each variable stores the area its box goes into.
* **Domains:** the areas a box can go into.
    * Scenario 1: A1, A2, A3, A4, A5
    * Scenario 2: A1, A2, A4, A5, or NONE (the box is not stored, because A3 is closed)
* **Constraints:** each area must stay within its limits:
    * number of boxes ≤ max boxes
    * total volume ≤ max volume
    * total weight ≤ max weight

**How the solver searches:** python-constraint tries areas box by box and backtracks when it hits a dead end. It uses the techniques from the notes: it picks the most constrained box first (the degree and MRV heuristics) and removes options that can no longer work (forward checking).

**One important setting:** by default python-constraint only checks a constraint once *all 30 boxes* have been placed. I set `assigned=False` so the limits are checked after every single placement. Without this, the solver did not finish within 2 minutes. With it, it takes milliseconds.

In [4]:
BOX_NAMES = list(BOXES)                                   # ["B1", "B2", ..., "B30"]
AREAS_S2 = {"A1": AREAS["A1"], "A2": AREAS["A2"], "A4": AREAS["A4"], "A5": AREAS["A5"]}


def within_limits(placements, areas):
    """Add up the boxes, volume and weight in each area and check they are within the limits."""
    totals = {}
    for area in areas:
        totals[area] = [0, 0, 0]                          # [boxes, volume, weight]

    for box, area in zip(BOX_NAMES, placements):
        if area in totals:                                # skip boxes not placed yet, or left out
            totals[area][0] += 1
            totals[area][1] += BOXES[box][0]
            totals[area][2] += BOXES[box][1]

    for area in areas:
        max_boxes, max_volume, max_weight = areas[area]
        boxes, volume, weight = totals[area]
        if boxes > max_boxes or volume > max_volume or weight > max_weight:
            return False
    return True


def scenario1_limits(*placements):
    return within_limits(placements, AREAS)


def scenario2_limits(*placements):
    return within_limits(placements, AREAS_S2)


def show_result(solution, areas):
    """Print which boxes went into each area, and the totals."""
    for area in areas:
        boxes_here = []
        for box in BOX_NAMES:
            if solution[box] == area:
                boxes_here.append(box)
        volume = sum(BOXES[b][0] for b in boxes_here)
        weight = sum(BOXES[b][1] for b in boxes_here)
        max_boxes, max_volume, max_weight = areas[area]
        print(f"{area}: {', '.join(boxes_here)}")
        print(f"    boxes {len(boxes_here)}/{max_boxes}   volume {volume}/{max_volume}   weight {weight}/{max_weight}")

    stored = [b for b in BOX_NAMES if solution[b] in areas]
    left_out = [b for b in BOX_NAMES if solution[b] == "NONE"]
    if left_out:
        print(f"Left out: {', '.join(left_out)}")
    print(f"TOTAL stored: {len(stored)} boxes, {sum(BOXES[b][1] for b in stored)} kg")

### Scenario 1 – all five areas open

In [5]:
problem = constraint.Problem()

# Variables: one per box. Domain: the five areas.
problem.addVariables(BOX_NAMES, ["A1", "A2", "A3", "A4", "A5"])

# Constraint: every area stays within its limits.
# assigned=False means the limits are checked after every box is placed, not only at the end.
problem.addConstraint(constraint.FunctionConstraint(scenario1_limits, assigned=False), BOX_NAMES)

start = time.time()
sol1 = problem.getSolution()
print(f"Solved in {time.time() - start:.4f} seconds\n")

if sol1:
    show_result(sol1, AREAS)
else:
    print("No solution")

Solved in 0.0055 seconds

A1: B6, B7, B8, B9
    boxes 4/7   volume 10/18   weight 140/260
A2: B3, B4, B5, B27, B28, B30
    boxes 6/6   volume 15/16   weight 250/250
A3: B21, B22, B23, B24, B25, B26, B29
    boxes 7/7   volume 15/18   weight 250/270
A4: B2, B16, B17, B18, B19, B20
    boxes 6/6   volume 15/15   weight 230/240
A5: B1, B10, B11, B12, B13, B14, B15
    boxes 7/7   volume 17/17   weight 260/260
TOTAL stored: 30 boxes, 1130 kg


### Scenario 2 – area A3 closed
With A3 closed, the other four areas can only hold:
* **26 boxes** (7 + 6 + 6 + 7)
* **1010 kg** (260 + 250 + 240 + 260)

In [7]:
areas2 = AREAS_S2



def left_out_ok(*placements):
    """Check the boxes left out (NONE) are within the targets."""
    count = 0
    weight = 0
    for box, area in zip(BOX_NAMES, placements):
        if area == "NONE":
            count += 1
            weight += BOXES[box][1]
    return count <= max_left_out and weight <= max_left_out_weight


# Step 1: work out the best possible result with simple arithmetic
total_weight = sum(weight for size, weight in BOXES.values())          # 1130 kg
max_boxes = sum(limits[0] for limits in AREAS_S2.values())             # 26 boxes
max_weight = sum(limits[2] for limits in AREAS_S2.values())            # 1010 kg
print(f"Best possible: {max_boxes} boxes and {max_weight} kg")
print(f"So at least {len(BOX_NAMES) - max_boxes} boxes and {total_weight - max_weight} kg must be left out\n")

# Step 2: ask the CSP solver, starting from the best possible result and working down


Best possible: 26 boxes and 1010 kg
So at least 4 boxes and 120 kg must be left out

